In [1]:
import os
import urllib
import tarfile
import glob

raw_dir = os.path.join(os.path.dirname('__file__'), '..', 'data', 'raw')
os.makedirs(raw_dir, exist_ok=True)

url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
file_path = os.path.join(raw_dir, "aclImdb_v1.tar.gz")

if not os.path.exists(file_path):
    urllib.request.urlretrieve(url, file_path)

if not os.path.exists(os.path.join(raw_dir, 'aclImdb')):
    with tarfile.open(file_path, 'r:gz') as tar:
        tar.extractall(path=raw_dir)

The data is dowloaded and saved to the directory ..\\data\\raw.

In [2]:
train_dir = os.path.join(raw_dir, 'aclImdb', 'train')
test_dir = os.path.join(raw_dir, 'aclImdb', 'test')

def join_data(search_path):
    texts = []
    labels = []
    i = 0
    flag = False
    for label in ['neg', 'pos']:
        folder = os.path.join(search_path, label)
        for path in glob.glob(os.path.join(folder, '*.txt')):
            with open(path, 'r', encoding='utf-8') as text:
                texts.append(text.read())
                labels.append(1 if label == 'pos' else 0)
    return texts, labels
            
X_train, y_train = join_data(train_dir)
X_test, y_test = join_data(test_dir)

Texts are gathered together to one list and save to directory ..\\data\\raw as test-texts.txt for testing set texts and train-texts.txt for training set.

In [3]:
print(f'Training set size: {len(X_train)}')
print(f'Test set size: {len(X_test)}')

Training set size: 25000
Test set size: 25000


In [4]:
print('Amount of neg class:', sum([1 for i in y_train if i == 0]))
print('Amount of pos class:', sum([1 for i in y_train if i == 1]))

Amount of neg class: 12500
Amount of pos class: 12500


Training set size equals testing set size equals 25 000. Total 50 000 objects. 

Classes are well balanced: 50% of positive texts and 50% of negative ones.

In [5]:
def save_data(file_name, path, data):
    with open(os.path.join(path, file_name), 'w', encoding='utf-8') as data_file:
        for text in data:
            data_file.write(text + '\n')

In [6]:
save_data('train-texts.txt', raw_dir, X_train) 
save_data('test-texts.txt', raw_dir, X_test)

In [7]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[-().,:;?!]", " ", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

X_train_cleaned = [clean_text(text) for text in X_train]
X_test_cleaned = [clean_text(text) for text in X_test]

clean_texts is a function for cleaning texts from HTML tags, extra space symbols, punctuation marks, and lowercase the words.

In [8]:
processed_dir = os.path.join(os.path.dirname('__file__'),  '..', 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)

save_data('cleaned-train.txt', processed_dir, X_train_cleaned)
save_data('cleaned-test.txt', processed_dir, X_test_cleaned)

save_data('train-labels.txt', processed_dir, [str(label) for label in y_train])
save_data('test-labels.txt', processed_dir, [str(label) for label in y_test])

The texts were cleaned manually in this notebook for the analysis by clean_tests function. However, then they will be cleaned within a TfidfVectorizer object.

In [9]:
print('Average number of words in training set:', round(sum([len(text.split()) for text in X_train_cleaned]) / len(X_train_cleaned)))
print('Average number of words in test set:', round(sum([len(text.split()) for text in X_test_cleaned]) / len(X_test_cleaned)))
print('Average number of symbols in training set:', round(sum([len(text) for text in X_train_cleaned]) / len(X_train_cleaned)))
print('Average number of symbols in test set:', round(sum([len(text) for text in X_test_cleaned]) / len(X_test_cleaned)))

Average number of words in training set: 232
Average number of words in test set: 227
Average number of symbols in training set: 1259
Average number of symbols in test set: 1229


In [10]:
print('Maximum number of words in training set:', max([len(text.split()) for text in X_train_cleaned]))
print('Maximum number of words in test set:', max([len(text.split()) for text in X_test_cleaned]))
print('Maximum number of symbols in training set:', max([len(text) for text in X_train_cleaned]))
print('Maximum number of symbols in test set:', max([len(text) for text in X_test_cleaned]))

Maximum number of words in training set: 2460
Maximum number of words in test set: 2234
Maximum number of symbols in training set: 13281
Maximum number of symbols in test set: 12259


In [11]:
print('Minimum number of words in training set:', min([len(text.split()) for text in X_train_cleaned]))
print('Minimum number of words in test set:', min([len(text.split()) for text in X_test_cleaned]))
print('Minimum number of symbols in training set:', min([len(text) for text in X_train_cleaned]))
print('Minimum number of symbols in test set:', min([len(text) for text in X_test_cleaned]))

Minimum number of words in training set: 10
Minimum number of words in test set: 6
Minimum number of symbols in training set: 51
Minimum number of symbols in test set: 30


There is a big range of lengths of comments. However, according to the average values, most of the comments are not so big. Also, training set's texts are longer than testing set's ones.

In [12]:
freq_train_dict = {}
for text in X_train_cleaned:
    for word in text.split():
        freq_train_dict[word] = freq_train_dict.get(word, 0) + 1

freq_train_list = []
for word, freq in freq_train_dict.items():
    freq_train_list.append((word, freq))

freq_train_list = sorted(freq_train_list, key=lambda x: x[1], reverse=True)

print('The most frequent words:')
i = 0
while i <= 10:
    print(f'{freq_train_list[i][0]}: {freq_train_list[i][1]}')
    i += 1

print('\nThe least frequent words:')
i -= 1
while i > 0:
    print(f'{freq_train_list[-i][0]}: {freq_train_list[-i][1]}')
    i -= 1

print('\nWords count:', len(freq_train_list))
print('\nCount of word with frequency = 1:', [x[1] for x in freq_train_list].count(1))

The most frequent words:
the: 336716
and: 163922
a: 162987
of: 145859
to: 135708
is: 107314
in: 93964
it: 79044
i: 77155
this: 75999
that: 69789

The least frequent words:
lowish: 1
hornes: 1
petunias: 1
inchworms: 1
billionare: 1
hightly: 1
slagged: 1
malkovitchesque: 1
muppified: 1
hued: 1

Words count: 80476

Count of word with frequency = 1: 31770


As expected the most frequent words are articles and prepositions.

But there is also a big fraction of words with the frequency of 1 despite the fact that there is quite big number of texts in training set. However, the absense of lemmatization should be taken into consideration.